# Data Cleaning

## 1. Introduction

This notebook prepares the merged CRMLS sales dataset for model development. It performs deterministic cleaning that can be completed before the chronological train/validation/test split.

The notebook:

- keeps only residential single-family homes;
- standardizes important data types and formats;
- removes exact duplicates and older versions of repeated `ListingKey` records;
- removes records with invalid target, date, or living-area values;
- converts invalid secondary feature values to missing values; and
- records the number and percentage of rows removed at each step.

The target variable is `ClosePrice`.

Imputation, encoding, scaling, and statistical outlier thresholds are intentionally postponed until after the chronological split so that they can be learned from the training data only.


## 2. Imports and Setup

### 2.1 Import Libraries


In [1]:
import pandas as pd
import numpy as np

### 2.2 Initialize the Cleaning Log

The helper function records the row count before and after each row-removal step. The log remains inside the notebook and does not create a separate audit file.


In [2]:
cleaning_log = []

def log_cleaning_step(step, rows_before, rows_after, reason):
    rows_removed = rows_before - rows_after

    cleaning_log.append({
        "Step": step,
        "Rows Before": rows_before,
        "Rows Removed": rows_removed,
        "Rows After": rows_after,
        "Percentage Removed": rows_removed / rows_before * 100,
        "Reason": reason
    })

## 3. Load and Inspect the Dataset

### 3.1 Load the Merged Data


In [3]:
file_path = "../data/merged_crmls_sold.csv"

df_raw = pd.read_csv(
    file_path,
    dtype={
        "ListingKey": "string",
        "ListingId": "string",
        "PostalCode": "string",
        "source_period": "string"
    },
    low_memory=False
)

print(f"Dataset shape: {df_raw.shape[0]:,} rows and {df_raw.shape[1]} columns")

Dataset shape: 794,271 rows and 84 columns


In [4]:
df_raw.head()

,source_period,source_file,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,MiddleOrJuniorSchoolDistrict,latfilled,lonfilled,BuyerAgentAOR,ListAgentAOR
0,2022-02,CRMLSSold20220101_20231231_filled.csv,"Carpet,Wood",True,NaN,NaN,True,98000.0,556366533,michellefsellsoc@gmail.com,...,0.0,ABC Unified,92708,0.0,NaN,NaN,False,False,NaN,NaN
1,2022-02,CRMLSSold20220101_20231231_filled.csv,NaN,False,NaN,NaN,False,1200.0,556366530,dineshcalre@gmail.com,...,1.0,Apple Valley Unified,92308,0.0,43000.0,NaN,False,False,NaN,NaN
2,2022-04,CRMLSSold20220101_20231231_filled.csv,NaN,True,NaN,NaN,False,1100000.0,556366044,cindydavishomes@gmail.com,...,1.0,Solana Beach,92075,370.0,NaN,NaN,False,False,NaN,NaN
3,2022-01,CRMLSSold20220101_20231231_filled.csv,NaN,True,NaN,NaN,False,2499999.0,556365765,bryanmeathe@gmail.com,...,2.0,Carlsbad Unified,92008,140.0,13376.0,NaN,False,False,NaN,NaN
4,2022-01,CRMLSSold20220101_20231231_filled.csv,"Carpet,Tile",NaN,NaN,NaN,NaN,598888.0,556365290,steven@westsideres.com,...,1.0,Other,95111,300.0,2738.0,NaN,False,False,NaN,NaN


In [5]:
df_raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 794271 entries, 0 to 794270
Data columns (total 84 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   source_period                 794271 non-null  string 
 1   source_file                   794271 non-null  object 
 2   Flooring                      465886 non-null  object 
 3   ViewYN                        707277 non-null  object 
 4   WaterfrontYN                  434 non-null     object 
 5   BasementYN                    13172 non-null   object 
 6   PoolPrivateYN                 690017 non-null  object 
 7   OriginalListPrice             791935 non-null  float64
 8   ListingKey                    794271 non-null  string 
 9   ListAgentEmail                792478 non-null  object 
 10  CloseDate                     794271 non-null  object 
 11  ClosePrice                    794263 non-null  float64
 12  ListAgentFirstName            789394 non-nul

### 3.3 Verify Required Columns

This check confirms that the columns needed for filtering, cleaning, and later model preparation are present.


In [6]:
important_columns = [
    "source_period",
    "source_file",
    "ListingKey",
    "CloseDate",
    "ClosePrice",
    "PropertyType",
    "PropertySubType",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "Latitude",
    "Longitude",
    "PostalCode"
]

missing_columns = [
    column for column in important_columns
    if column not in df_raw.columns
]

if len(missing_columns) == 0:
    print("All important columns are available.")
else:
    print("Missing columns:", missing_columns)

All important columns are available.


In [7]:
df_raw[important_columns].head()

,source_period,source_file,ListingKey,CloseDate,ClosePrice,PropertyType,PropertySubType,LivingArea,BedroomsTotal,BathroomsTotalInteger,LotSizeSquareFeet,YearBuilt,Latitude,Longitude,PostalCode
0,2022-02,CRMLSSold20220101_20231231_filled.csv,556366533,2022-02-25,95000.0,ManufacturedInPark,NaN,1368.0,2.0,2.0,NaN,1969.0,33.699517,-117.960655,92708
1,2022-02,CRMLSSold20220101_20231231_filled.csv,556366530,2022-02-19,1200.0,ResidentialLease,Apartment,850.0,2.0,1.0,43000.0,1987.0,34.497691,-117.192323,92308
2,2022-04,CRMLSSold20220101_20231231_filled.csv,556366044,2022-04-15,1100000.0,Residential,Townhouse,1344.0,3.0,3.0,NaN,1974.0,32.981292,-117.262529,92075
3,2022-01,CRMLSSold20220101_20231231_filled.csv,556365765,2022-01-04,2499999.0,Residential,SingleFamilyResidence,2645.0,4.0,4.0,13376.0,2016.0,33.147270,-117.340604,92008
4,2022-01,CRMLSSold20220101_20231231_filled.csv,556365290,2022-01-12,640000.0,Residential,Townhouse,1198.0,2.0,3.0,2738.0,1989.0,37.295089,-121.842471,95111


## 4. Filter the Modeling Scope

### 4.1 Inspect Property Categories


In [8]:
df_raw["PropertyType"].value_counts(dropna=False)

PropertyType
Residential            532920
ResidentialLease       181556
Land                    27429
ManufacturedInPark      21429
ResidentialIncome       21015
CommercialSale           5164
CommercialLease          4205
BusinessOpportunity       552
Resid                       1
Name: count, dtype: int64

In [9]:
df_raw["PropertySubType"].value_counts(dropna=False)

PropertySubType
SingleFamilyResidence    484815
Condominium              137769
NaN                       62466
Townhouse                 45583
Apartment                 17822
Duplex                    14860
ManufacturedOnLand         8058
Triplex                    4560
Quadruplex                 4537
StockCooperative           2681
MixedUse                   2609
Office                     2235
Retail                     1478
Industrial                  931
Cabin                       648
Studio                      570
Business                    533
Warehouse                   380
RoomingHouse                376
SpecialPurpose              215
MultiFamily                 183
BoatSlip                    149
Agriculture                 123
OwnYourOwn                  112
UnimprovedLand              102
Loft                        100
MobileHome                   96
WaterPositionWithLand        93
ManufacturedHome             68
Farm                         30
Timeshare               

### 4.2 Keep Residential Single-Family Homes

The project only records where `PropertyType` is `Residential` and `PropertySubType` is `SingleFamilyResidence`. This is a scope filter rather than a data-quality deletion.


In [10]:
df = df_raw.copy()

df["PropertyType"] = (
    df["PropertyType"]
    .astype("string")
    .str.strip()
)

df["PropertySubType"] = (
    df["PropertySubType"]
    .astype("string")
    .str.strip()
)

rows_before = len(df)

df = df[
    (df["PropertyType"] == "Residential")
    & (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

df = df.reset_index(drop=True)

rows_after = len(df)
rows_removed = rows_before - rows_after

print(f"Rows before filtering: {rows_before:,}")
print(f"Rows after filtering:  {rows_after:,}")
print(f"Rows removed:          {rows_removed:,}")
print(f"Percentage retained:   {rows_after / rows_before:.2%}")

log_cleaning_step(
    step="Scope filter: Residential SFR",
    rows_before=rows_before,
    rows_after=rows_after,
    reason="Keep only Residential SingleFamilyResidence required by the project"
)

Rows before filtering: 794,271
Rows after filtering:  399,157
Rows removed:          395,114
Percentage retained:   50.25%


In [11]:
print(df[["PropertyType", "PropertySubType"]].drop_duplicates())
print("Dataset shape:", df.shape)

  PropertyType        PropertySubType
0  Residential  SingleFamilyResidence
Dataset shape: (399157, 84)


**Section result:** The dataset is restricted to the property type required by the project.


## 5. Standardize Data Types and Formats

These steps standardize dates, numeric variables, text fields, ZIP codes, and month variables. No rows are removed in this section.

### 5.1 Convert Date Columns


In [12]:
date_columns = [
    "CloseDate",
    "ListingContractDate",
    "PurchaseContractDate",
    "ContractStatusChangeDate"
]

for column in date_columns:
    if column in df.columns:
        df[column] = pd.to_datetime(
            df[column],
            errors="coerce"
        )

df[date_columns].dtypes

CloseDate                   datetime64[ns]
ListingContractDate         datetime64[ns]
PurchaseContractDate        datetime64[ns]
ContractStatusChangeDate    datetime64[ns]
dtype: object

### 5.2 Create the Source Month

`source_month` is used to identify the most recent version when the same `ListingKey` appears in more than one source snapshot.


In [13]:
df["source_month"] = pd.to_datetime(df["source_period"], format="%Y-%m", errors="coerce")

df[["source_period", "source_month"]].head()

,source_period,source_month
0,2022-01,2022-01-01
1,2022-01,2022-01-01
2,2022-03,2022-03-01
3,2022-01,2022-01-01
4,2022-01,2022-01-01


### 5.3 Convert Numeric Columns


In [14]:
numeric_columns = [
    "ClosePrice",
    "LivingArea",
    "BedroomsTotal",
    "BathroomsTotalInteger",
    "LotSizeSquareFeet",
    "YearBuilt",
    "Latitude",
    "Longitude",
    "GarageSpaces",
    "ParkingTotal",
    "Stories",
    "AssociationFee"
]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce"
        )

df[numeric_columns].dtypes

ClosePrice               float64
LivingArea               float64
BedroomsTotal            float64
BathroomsTotalInteger    float64
LotSizeSquareFeet        float64
YearBuilt                float64
Latitude                 float64
Longitude                float64
GarageSpaces             float64
ParkingTotal             float64
Stories                  float64
AssociationFee           float64
dtype: object

### 5.4 Clean String Columns

Leading and trailing spaces are removed, and empty strings are converted to missing values.


In [15]:
string_columns = [
    "ListingKey",
    "ListingId",
    "PostalCode",
    "City",
    "CountyOrParish",
    "MLSAreaMajor",
    "HighSchoolDistrict",
    "PropertyType",
    "PropertySubType",
    "source_period",
    "source_file"
]

for column in string_columns:
    if column in df.columns:
        df[column] = (df[column].astype("string").str.strip().replace("", pd.NA))

df[string_columns].dtypes

ListingKey            string[python]
ListingId             string[python]
PostalCode            string[python]
City                  string[python]
CountyOrParish        string[python]
MLSAreaMajor          string[python]
HighSchoolDistrict    string[python]
PropertyType          string[python]
PropertySubType       string[python]
source_period         string[python]
source_file           string[python]
dtype: object

### 5.5 Standardize ZIP Codes

Only the first five digits are retained so that standard ZIP codes and ZIP+4 values use the same format.


In [16]:
df["PostalCode"] = (df["PostalCode"].str.extract(r"(\d{5})", expand=False))

# check
df["PostalCode"].str.len().value_counts(dropna=False)

PostalCode
5       399155
<NA>         2
Name: count, dtype: Int64

### 5.6 Create the Closing Month


In [17]:
df["close_month"] = (df["CloseDate"].dt.to_period("M").astype("string"))

df[ ["CloseDate", "close_month"] ].head()

,CloseDate,close_month
0,2022-01-04,2022-01
1,2022-01-10,2022-01
2,2022-03-23,2022-03
3,2022-01-10,2022-01
4,2022-01-19,2022-01


### 5.7 Validate the Conversions


In [18]:
print("Dataset shape:", df.shape)

print("\nClose date range:")
print(df["CloseDate"].min(), "to", df["CloseDate"].max())

print("\nSource month range:")
print(df["source_month"].min(), "to", df["source_month"].max())

Dataset shape: (399157, 86)

Close date range:
2022-01-01 00:00:00 to 2026-05-31 00:00:00

Source month range:
2022-01-01 00:00:00 to 2026-05-01 00:00:00


In [19]:
conversion_check = pd.DataFrame({
    "column": [

        "CloseDate",
        "source_month",
        "ClosePrice",
        "LivingArea",
        "BedroomsTotal",
        "BathroomsTotalInteger",
        "LotSizeSquareFeet",
        "YearBuilt",
        "Latitude",
        "Longitude",
        "PostalCode"
    
    ],

    "missing_count": [

        df["CloseDate"].isna().sum(),
        df["source_month"].isna().sum(),
        df["ClosePrice"].isna().sum(),
        df["LivingArea"].isna().sum(),
        df["BedroomsTotal"].isna().sum(),
        df["BathroomsTotalInteger"].isna().sum(),
        df["LotSizeSquareFeet"].isna().sum(),
        df["YearBuilt"].isna().sum(),
        df["Latitude"].isna().sum(),
        df["Longitude"].isna().sum(),
        df["PostalCode"].isna().sum()
    
    ]
})

conversion_check["missing_percentage"] = (conversion_check["missing_count"] / len(df) * 100)
conversion_check

,column,missing_count,missing_percentage
0,CloseDate,0,0.000000
1,source_month,0,0.000000
2,ClosePrice,2,0.000501
3,LivingArea,210,0.052611
4,BedroomsTotal,0,0.000000
5,BathroomsTotalInteger,75,0.018790
6,LotSizeSquareFeet,6826,1.710104
7,YearBuilt,308,0.077163
8,Latitude,193,0.048352
9,Longitude,193,0.048352


**Section result:** Important fields now use consistent data types and formats, while the row count remains unchanged.


## 6. Remove Duplicate Records

### 6.1 Remove Exact Duplicate Business Records

Source columns are excluded from the comparison because they describe file provenance rather than the underlying property record. One copy of each exact duplicate business record is retained.


In [20]:
source_columns = [
    "source_file",
    "source_period",
    "source_month"
]

comparison_columns = [
    column for column in df.columns
    if column not in source_columns
]

exact_duplicate_count = df.duplicated(
    subset=comparison_columns
).sum()

print(f"Exact duplicate rows: {exact_duplicate_count:,}")

Exact duplicate rows: 31


In [21]:
rows_before = len(df)

df = (
    df.drop_duplicates(
        subset=comparison_columns,
        keep="first"
    )
    .reset_index(drop=True)
)

rows_after = len(df)

print(f"Rows before:  {rows_before:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Rows after:   {rows_after:,}")


# log cleaning steps
log_cleaning_step(
    step="Remove exact duplicate records",
    rows_before=rows_before,
    rows_after=rows_after,
    reason="Remove identical business records"
)

Rows before:  399,157
Rows removed: 31
Rows after:   399,126


**Result:** Exact duplicate business records are removed before checking repeated listing identifiers.


### 6.2 Keep the Latest Version of Each `ListingKey`

The same `ListingKey` may appear in multiple source snapshots. Records are sorted by source month, the latest version is retained, and rows with a missing `ListingKey` are preserved.


In [22]:
duplicate_key_mask = (
    df["ListingKey"].notna()
    & df["ListingKey"].duplicated(keep=False)
)

duplicate_listing_rows = df.loc[duplicate_key_mask]

print(
    "Rows belonging to duplicate ListingKeys:",
    len(duplicate_listing_rows)
)

print(
    "Number of duplicate ListingKeys:",
    duplicate_listing_rows["ListingKey"].nunique()
)

print(
    "Extra rows to remove:",
    len(duplicate_listing_rows) - duplicate_listing_rows["ListingKey"].nunique()
)

Rows belonging to duplicate ListingKeys: 486
Number of duplicate ListingKeys: 241
Extra rows to remove: 245


In [23]:
rows_before = len(df)

df_with_key = (
    df[df["ListingKey"].notna()]
    .sort_values(["source_month", "source_file"])
    .drop_duplicates(
        subset="ListingKey",
        keep="last"
    )
)

df_without_key = df[df["ListingKey"].isna()]

df = pd.concat(
    [df_with_key, df_without_key],
    ignore_index=True
)

rows_after = len(df)

print(f"Rows removed by ListingKey: {rows_before - rows_after:,}")
print(f"Rows remaining: {rows_after:,}")

log_cleaning_step(
    step="Remove older duplicate ListingKey versions",
    rows_before=rows_before,
    rows_after=rows_after,
    reason="Keep the latest source month for each non-missing ListingKey"
)

Rows removed by ListingKey: 245
Rows remaining: 398,881


In [24]:
remaining_duplicates = (
    df.loc[df["ListingKey"].notna(), "ListingKey"]
    .duplicated()
    .sum()
)

print("Remaining duplicate ListingKeys:", remaining_duplicates)

Remaining duplicate ListingKeys: 0


**Section result:** Each non-missing `ListingKey` now appears only once, with the latest available version retained.


## 7. Remove Invalid Core Records

Core records are removed when the target or living area is unusable, the closing date is missing, or the closing date occurs before the listing contract date.


In [25]:
rows_before = len(df)

valid_rows = (
    df["ClosePrice"].notna()
    & (df["ClosePrice"] > 0)
    & df["CloseDate"].notna()
    & df["LivingArea"].notna()
    & (df["LivingArea"] > 0)
    & (
        df["ListingContractDate"].isna() | (df["CloseDate"] >= df["ListingContractDate"])
    )
)

df = (
    df.loc[valid_rows]
    .reset_index(drop=True)
)

rows_after = len(df)

print(f"Rows before:  {rows_before:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Rows after:   {rows_after:,}")

log_cleaning_step(
    step="Remove invalid core records",
    rows_before=rows_before,
    rows_after=rows_after,
    reason="Invalid ClosePrice, CloseDate, LivingArea, or date relationship"
)

Rows before:  398,881
Rows removed: 420
Rows after:   398,461


**Section result:** Records that cannot support reliable model training are removed.


## 8. Replace Invalid Feature Values with Missing Values

Invalid secondary feature values are converted to `NaN` instead of deleting the entire property record. Missing values will be handled later by preprocessing fitted on the training data only.

### 8.1 Define and Apply Validity Rules


In [26]:
invalid_conditions = {
    "LotSizeSquareFeet": (
        df["LotSizeSquareFeet"].notna()
        & (df["LotSizeSquareFeet"] <= 0)
    ),

    "YearBuilt": (
        df["YearBuilt"].notna()
        & (
            (df["YearBuilt"] <= 0)
            | (df["YearBuilt"] > df["CloseDate"].dt.year)
        )
    ),

    "Latitude": (
        df["Latitude"].notna()
        & ~df["Latitude"].between(32, 42.5)
    ),

    "Longitude": (
        df["Longitude"].notna()
        & ~df["Longitude"].between(-125, -113)
    ),

    "BedroomsTotal": (
        df["BedroomsTotal"].notna()
        & (df["BedroomsTotal"] < 0)
    ),

    "BathroomsTotalInteger": (
        df["BathroomsTotalInteger"].notna()
        & (df["BathroomsTotalInteger"] < 0)
    ),

    "GarageSpaces": (
        df["GarageSpaces"].notna()
        & (df["GarageSpaces"] < 0)
    ),

    "ParkingTotal": (
        df["ParkingTotal"].notna()
        & (df["ParkingTotal"] < 0)
    ),

    "Stories": (
        df["Stories"].notna()
        & (df["Stories"] <= 0)
    ),

    "AssociationFee": (
        df["AssociationFee"].notna()
        & (df["AssociationFee"] < 0)
    )
}

In [27]:
summary_data = []

for column, condition in invalid_conditions.items():
    changed_count = int(condition.sum())

    summary_data.append({
        "Column Name": column,
        "Invalid Values Changed to NaN": changed_count
    })

    df.loc[condition, column] = np.nan

invalid_value_summary = pd.DataFrame(summary_data)

display(invalid_value_summary)

print("-" * 50)
print(f"Total rows remaining: {len(df):,}")

,Column Name,Invalid Values Changed to NaN
0,LotSizeSquareFeet,214
1,YearBuilt,35
2,Latitude,59
3,Longitude,88
4,BedroomsTotal,0
5,BathroomsTotalInteger,0
6,GarageSpaces,0
7,ParkingTotal,102
8,Stories,0
9,AssociationFee,0


--------------------------------------------------
Total rows remaining: 398,461


### 8.2 Keep Coordinates as a Complete Pair

If either latitude or longitude is missing, both values are set to missing so that an incomplete coordinate pair is not used later.


In [28]:
incomplete_coordinates = (
    df["Latitude"].isna()
    | df["Longitude"].isna()
)

print(
    "Rows with incomplete coordinates:",
    incomplete_coordinates.sum()
)

df.loc[
    incomplete_coordinates,
    ["Latitude", "Longitude"]
] = np.nan

print(f"Total rows remaining: {len(df):,}")

Rows with incomplete coordinates: 287
Total rows remaining: 398,461


**Section result:** Invalid secondary values are represented as missing values, and no additional rows are removed.


## 9. Cleaning Summary


In [29]:
cleaning_log_df = pd.DataFrame(cleaning_log)

display_df = cleaning_log_df.copy()

display_df["Rows Before"] = display_df["Rows Before"].map("{:,.0f}".format)
display_df["Rows Removed"] = display_df["Rows Removed"].map("{:,.0f}".format)
display_df["Rows After"] = display_df["Rows After"].map("{:,.0f}".format)
display_df["Percentage Removed"] = (
    display_df["Percentage Removed"]
    .map("{:.4f}%".format)
)

display(display_df)

in_scope_rows = cleaning_log_df.loc[
    cleaning_log_df["Step"] == "Scope filter: Residential SFR",
    "Rows After"
].iloc[0]

quality_rows_removed = cleaning_log_df.loc[
    cleaning_log_df["Step"] != "Scope filter: Residential SFR",
    "Rows Removed"
].sum()

print(
    f"Rows removed for data-quality reasons after scope filtering: "
    f"{quality_rows_removed:,.0f} "
    f"({quality_rows_removed / in_scope_rows:.4%})"
)

print(f"Final cleaned rows: {len(df):,}")


,Step,Rows Before,Rows Removed,Rows After,Percentage Removed,Reason
0,Scope filter: Residential SFR,"794,271","395,114","399,157",49.7455%,Keep only Residential SingleFamilyResidence re...
1,Remove exact duplicate records,"399,157",31,"399,126",0.0078%,Remove identical business records
2,Remove older duplicate ListingKey versions,"399,126",245,"398,881",0.0614%,Keep the latest source month for each non-miss...
3,Remove invalid core records,"398,881",420,"398,461",0.1053%,"Invalid ClosePrice, CloseDate, LivingArea, or ..."


Rows removed for data-quality reasons after scope filtering: 696 (0.1744%)
Final cleaned rows: 398,461


## 10. Conclusion

The dataset has been restricted to the required property type, standardized, deduplicated, and cleaned of logically invalid records. Invalid secondary feature values are stored as missing values rather than causing unnecessary row deletion.

The cleaned dataset is now ready for chronological train, validation, and test splitting. Imputation, encoding, scaling, and statistical outlier filtering should be fitted using the training data only.


In [30]:
from pathlib import Path

output_path = Path(
    "../data/processed/crmls_sfr_cleaned_base.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

df.to_csv(
    output_path,
    index=False
)

print(f"Cleaned dataset saved to: {output_path}")
print(f"Saved rows: {len(df):,}")
print(f"Saved columns: {df.shape[1]}")

Cleaned dataset saved to: ..\data\processed\crmls_sfr_cleaned_base.csv
Saved rows: 398,461
Saved columns: 86
